In [1]:
import heapq

N = 3  # puzzle size (3x3)

class Node:
    def __init__(self, matrix, x, y, level, parent=None):
        self.parent = parent
        self.matrix = [row[:] for row in matrix]  # deep copy
        self.x = x
        self.y = y
        self.level = level
        self.cost = 0  # heuristic cost

    def __lt__(self, other):
        return (self.cost + self.level) < (other.cost + other.level)


def print_matrix(matrix):
    for row in matrix:
        print(" ".join(map(str, row)))
    print()


def calculate_manhattan(matrix, goal):
    """Heuristic: sum of Manhattan distances of tiles from goal positions"""
    dist = 0
    pos = {}
    for i in range(N):
        for j in range(N):
            pos[goal[i][j]] = (i, j)

    for i in range(N):
        for j in range(N):
            val = matrix[i][j]
            if val != 0:
                goal_i, goal_j = pos[val]
                dist += abs(i - goal_i) + abs(j - goal_j)
    return dist


def is_safe(x, y):
    return 0 <= x < N and 0 <= y < N


def print_path(node):
    if node is None:
        return
    print_path(node.parent)
    print_matrix(node.matrix)


def solve(initial, x, y, goal):
    # Priority queue (min-heap)
    pq = []
    root = Node(initial, x, y, 0, None)
    root.cost = calculate_manhattan(initial, goal)
    heapq.heappush(pq, root)

    visited = set()
    visited.add(str(root.matrix))

    # directions: down, left, up, right
    row = [1, 0, -1, 0]
    col = [0, -1, 0, 1]

    while pq:
        min_node = heapq.heappop(pq)

        # goal reached
        if min_node.cost == 0:
            print("Solution path:")
            print_path(min_node)
            return

        # explore neighbors
        for i in range(4):
            new_x, new_y = min_node.x + row[i], min_node.y + col[i]
            if is_safe(new_x, new_y):
                child = Node(min_node.matrix, min_node.x, min_node.y, min_node.level + 1, min_node)
                # swap blank with neighbor
                child.matrix[min_node.x][min_node.y], child.matrix[new_x][new_y] = child.matrix[new_x][new_y], child.matrix[min_node.x][min_node.y]
                child.x, child.y = new_x, new_y
                child.cost = calculate_manhattan(child.matrix, goal)

                state_str = str(child.matrix)
                if state_str not in visited:
                    visited.add(state_str)
                    heapq.heappush(pq, child)


if __name__ == "__main__":
    # initial configuration (0 = blank)
    initial = [
        [0, 1, 2],
        [4, 5, 8],
        [6, 7, 3]
    ]

    # goal configuration
    goal = [
        [0, 3, 7],
        [4, 2, 5],
        [6, 8, 1]
    ]

    # blank tile position in initial
    x, y = 0, 0

    solve(initial, x, y, goal)


Solution path:
0 1 2
4 5 8
6 7 3

4 1 2
0 5 8
6 7 3

4 1 2
5 0 8
6 7 3

4 0 2
5 1 8
6 7 3

4 2 0
5 1 8
6 7 3

4 2 8
5 1 0
6 7 3

4 2 8
5 1 3
6 7 0

4 2 8
5 1 3
6 0 7

4 2 8
5 0 3
6 1 7

4 2 8
5 3 0
6 1 7

4 2 0
5 3 8
6 1 7

4 0 2
5 3 8
6 1 7

4 3 2
5 0 8
6 1 7

4 3 2
5 8 0
6 1 7

4 3 2
5 8 7
6 1 0

4 3 2
5 8 7
6 0 1

4 3 2
5 0 7
6 8 1

4 3 2
0 5 7
6 8 1

0 3 2
4 5 7
6 8 1

3 0 2
4 5 7
6 8 1

3 2 0
4 5 7
6 8 1

3 2 7
4 5 0
6 8 1

3 2 7
4 0 5
6 8 1

3 0 7
4 2 5
6 8 1

0 3 7
4 2 5
6 8 1

